# Kokoro 音色蒸馏 - 梯度反演 style vector

基于论文 *Extracting Voice Styles from Frozen TTS Models via Gradient-Based Inverse Optimization* (arXiv:2607.25351)。

## 原理

**之前失败的方案**：
- mel L1 → 静音坍缩（静音帧主导损失）
- + 静音惩罚 → 有声音但音色不变（mel L1 不提供音色方向梯度）
- StyleTTS2 encoder → 杂音（分布不匹配，OOD）

**本方案**：
```
目标音频 → WavLM L4 → 目标统计量 (mean, std)
                                ↑ 损失
style_vec (256维, 可训练) → Kokoro decoder → 合成音频 → WavLM L4 → 合成统计量
                                ↓
                          反向传播更新 style_vec
```

关键点：WavLM 第 4 层是 speaker ID 峰值层，time-pooled 统计量编码音色而非内容。256 维 vs 82M 参数，是反问题而非学习问题。

## 使用方法

1. 将 `ref3_tts_dataset_routed_500.zip` 上传为 Kaggle Dataset
2. Notebook Settings: Internet=ON, Accelerator=GPU T4
3. Add Data → 添加该 Dataset
4. Run All

## 1. 环境准备

In [ ]:
# 安装依赖
!pip install kokoro transformers librosa soundfile pypinyin ordered_set cn2an speechbrain -q

In [ ]:
import os
import sys
import time
import json
import zipfile
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import librosa
import soundfile as sf

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. 定位数据集

Kaggle 上传 zip 后会自动解压，数据集根目录直接是 wav 文件。
自动扫描 `/kaggle/input/` 查找包含 wav 文件的目录，无需固定 dataset 名字。

In [ ]:
# 自动扫描 /kaggle/input/ 查找 wav 文件目录
# Kaggle 上传 zip 后会自动解压，根目录直接是 wav 文件
WAVS_DIR = None
best_count = 0

if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        wavs = [f for f in files if f.endswith('.wav')]
        if len(wavs) > best_count:
            best_count = len(wavs)
            WAVS_DIR = root

# 兼容本地测试
if WAVS_DIR is None:
    local_candidates = [
        '../ref3_tts_dataset_routed_500/wavs',
        './wavs',
    ]
    for d in local_candidates:
        if os.path.isdir(d):
            wavs = [f for f in os.listdir(d) if f.endswith('.wav')]
            if len(wavs) > 10:
                WAVS_DIR = d
                best_count = len(wavs)
                break

if WAVS_DIR is None:
    print('ERROR: No wav files found in /kaggle/input/')
    print('Available inputs:')
    if os.path.exists('/kaggle/input'):
        for item in os.listdir('/kaggle/input'):
            p = os.path.join('/kaggle/input', item)
            if os.path.isdir(p):
                files = os.listdir(p)
                print(f'  {p}/ ({len(files)} files)')
                for f in files[:5]:
                    print(f'    {f}')
    raise FileNotFoundError('No wav files found')

print(f'WAVS_DIR: {WAVS_DIR}')
print(f'Wav files: {best_count}')
all_wavs = sorted([os.path.join(WAVS_DIR, f) for f in os.listdir(WAVS_DIR) if f.endswith('.wav')])
print(f'First 3: {[os.path.basename(w) for w in all_wavs[:3]]}')

# 查找 manifest.jsonl（wav -> text 映射）
# 训练时使用数据集真实文本（含标点），保证文本与音频一一对应，
# 这样模型才能学到'标点位置应该静音'的规律。
MANIFEST_PATH = None
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'manifest.jsonl' in files:
            MANIFEST_PATH = os.path.join(root, 'manifest.jsonl')
            break
if MANIFEST_PATH is None:
    # 本地：manifest 与 wavs/ 同级
    parent = os.path.dirname(WAVS_DIR.rstrip('/\\'))
    cand = os.path.join(parent, 'manifest.jsonl')
    if os.path.isfile(cand):
        MANIFEST_PATH = cand
if MANIFEST_PATH:
    print(f'MANIFEST: {MANIFEST_PATH}')
else:
    print('WARNING: manifest.jsonl not found, training will fall back to fixed texts')


## 3. 加载模型

加载 Kokoro（PyTorch 原生模型）和 WavLM-Large，全部冻结。

In [ ]:
# 加载 Kokoro
from kokoro import KModel, KPipeline
print('Loading Kokoro model...')
model = KModel().to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)
print(f'Kokoro loaded, frozen, on {DEVICE}')
# 禁用 dropout：differentiable_forward 需要 model.train()（cudnn RNN
# backward 仅 train 模式可用），但 train 模式会激活 TextEncoder 与
# AdainResBlk1d 的 dropout，导致 F0/N 输出每次 forward 随机采样，
# loss 永远有噪声基线（无法降到 0）、early stopping 被噪声干扰。
# 将 dropout 概率置 0：保持 train 模式（RNN backward 可用）且输出确定性。
for m in model.modules():
    if isinstance(m, torch.nn.Dropout):
        m.p = 0.0
print('Dropout disabled (deterministic F0/N)')
# voices 在 KPipeline 上加载，不在 KModel 上
print(f'Context length: {model.context_length}')

In [ ]:
# 加载 ECAPA-TDNN speaker embedding（音色匹配）
# 数据集由 qwen3-tts 生成（与 Kokoro 不同源），WavLM 统计量无法区分
# 二者音色。ECAPA-TDNN 是 speaker verification 专用模型，跨 TTS
# 引擎区分可靠，且整条链路可微（梯度可流回 decoder -> style）。
from speechbrain.pretrained import EncoderClassifier
print('Loading ECAPA-TDNN...')
spk_enc = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir='/kaggle/working/spkrec-ecapa',
    run_opts={'device': str(DEVICE)}
)
# 冻结 ECAPA 参数（只作为特征提取器，梯度穿过但不更新）
for p in spk_enc.mods.embedding_model.parameters():
    p.requires_grad_(False)
print(f'ECAPA-TDNN loaded on {DEVICE}')

In [ ]:
# 创建 G2P pipeline（中文）
pipeline = KPipeline(lang_code='z', model=False)
print('G2P pipeline ready')

# 测试 G2P
test_text = '主人，今天天气真好。'
chunks = list(pipeline(test_text, voice='zf_xiaoyi'))
print(f'G2P test: {test_text} -> {repr(chunks[0].phonemes)}')

## 4. 定义可微 forward

Kokoro 原版 `forward_with_tokens` 有 `@torch.no_grad()` 装饰器，这里重写以保留梯度。

In [ ]:
def differentiable_forward(model, phonemes, ref_s, speed=1.0):
    """可微的 Kokoro forward，绕过 @torch.no_grad。
    
    Args:
        model: KModel 实例（参数已冻结）
        phonemes: 音素字符串
        ref_s: (1, 256) style vector, requires_grad=True
        speed: 语速
    Returns:
        audio:    (T,) 合成音频，保留梯度（依赖 acoustic 前128维 + F0/N）
        F0_pred:  (T_frm,) F0 轮廓，保留梯度（只依赖 prosody 后128维）
        N_pred:   (T_frm,) 能量轮廓，保留梯度（只依赖 prosody 后128维）
        duration: (T_token,) 每个音素的连续时长（round 前，可微）
        pred_dur: (T_token,) 每个音素的整数时长（round 后，用于对齐）
        input_ids:(1, T_token) token IDs（含 BOS/EOS）
    """
    device = ref_s.device

    # 临时切换到 train 模式：cudnn RNN backward 只能在 train 模式下调用
    # 参数仍然冻结 (requires_grad=False)，不会更新模型权重
    was_training = model.training
    model.train()

    # phonemes -> input_ids
    input_ids = list(filter(lambda i: i is not None,
                            map(lambda p: model.vocab.get(p), phonemes)))
    assert len(input_ids) + 2 <= model.context_length, \
        f'phoneme length {len(input_ids)+2} > context {model.context_length}'
    input_ids = torch.LongTensor([[0, *input_ids, 0]]).to(device)
    
    input_lengths = torch.full(
        (input_ids.shape[0],), input_ids.shape[-1],
        device=device, dtype=torch.long
    )
    
    text_mask = torch.arange(input_lengths.max()).unsqueeze(0).expand(
        input_lengths.shape[0], -1
    ).type_as(input_lengths)
    text_mask = torch.gt(text_mask + 1, input_lengths.unsqueeze(1)).to(device)
    
    # --- BERT encoder ---
    bert_dur = model.bert(input_ids, attention_mask=(~text_mask).int())
    d_en = model.bert_encoder(bert_dur).transpose(-1, -2)
    
    # --- prosody style (后128维) ---
    s = ref_s[:, 128:]
    
    # --- duration predictor ---
    d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
    x, _ = model.predictor.lstm(d)
    duration = model.predictor.duration_proj(x)
    duration = torch.sigmoid(duration).sum(axis=-1) / speed
    pred_dur = torch.round(duration).clamp(min=1).long().squeeze()
    
    indices = torch.repeat_interleave(
        torch.arange(input_ids.shape[1], device=device), pred_dur
    )
    pred_aln_trg = torch.zeros(
        (input_ids.shape[1], indices.shape[0]), device=device
    )
    pred_aln_trg[indices, torch.arange(indices.shape[0])] = 1
    pred_aln_trg = pred_aln_trg.unsqueeze(0).to(device)
    
    en = d.transpose(-1, -2) @ pred_aln_trg
    F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
    
    # --- text encoder ---
    t_en = model.text_encoder(input_ids, input_lengths, text_mask)
    asr = t_en @ pred_aln_trg
    
    # --- decoder (acoustic style: 前128维) ---
    audio = model.decoder(asr, F0_pred, N_pred, ref_s[:, :128]).squeeze()

    # 恢复原来的 eval 模式
    if not was_training:
        model.eval()

    return audio, F0_pred.squeeze(), N_pred.squeeze(), \
        duration.squeeze(), pred_dur, input_ids


# ============================================================
# 时间维度对齐辅助函数
# ============================================================

def token_level_mean(seq, dur):
    """将帧级序列按每个音素的时长聚合为 token 级均值。

    Kokoro 的 F0Ntrain 输出（F0_pred / N_pred）经过 ConvTranspose1d
    上采样，长度是 sum(dur) 的整数倍（通常 2x）。这里先用可微的
    线性插值将 seq 规整到 sum(dur) 长度，再按 dur 聚合。

    Args:
        seq: (T_seq,) 帧级序列（如 F0_pred / N_pred）
        dur: (T_token,) 每个音素的帧时长（整数）
    Returns:
        (T_token,) 每个音素区间内 seq 的均值
    """
    n = dur.shape[0]
    T_frame = int(dur.sum())
    T_seq = seq.shape[0]
    if T_seq != T_frame:
        # 分辨率不匹配：线性插值到 sum(dur) 长度（保留梯度）
        seq = torch.nn.functional.interpolate(
            seq.view(1, 1, T_seq), size=T_frame, mode='linear', align_corners=False
        ).view(-1)
    t_idx = torch.repeat_interleave(torch.arange(n, device=seq.device), dur)
    sums = torch.zeros(n, device=seq.device).scatter_add_(0, t_idx, seq)
    return sums / dur.float().clamp(min=1)


def ecapa_embedding(spk_enc, audio_16k, normalize=True):
    """ECAPA-TDNN 说话人嵌入（192 维）。

    可微：梯度可流回 audio -> decoder -> style（acoustic 前128维）。
    Args:
        spk_enc: EncoderClassifier 实例（参数已冻结）
        audio_16k: (1, T) 16kHz 音频
        normalize: 是否 L2 归一化
    Returns:
        (192,) 说话人嵌入
    """
    emb = spk_enc.encode_batch(audio_16k)   # (1, 1, 192)
    emb = emb.squeeze(0).squeeze(0)         # (192,)
    if normalize:
        emb = F.normalize(emb, dim=0)
    return emb


def spectral_band_profile(audio, sr=24000, n_fft=1024, hop=256,
                          n_bands=10):
    """可微的频谱频带能量分布（log 频率分桶，归一化占比，与时长无关）。

    尖细/清脆音色 → 高频频带能量占比高。用于把合成音色的频谱质感
    （而非仅 speaker embedding）拉向目标，解决"音色像但成熟/机械"。
    Args:
        audio: (T,) 波形（24kHz）
    Returns:
        (n_bands,) 归一化频带能量占比（和为 1，保留梯度）
    """
    if audio.dim() == 2:
        audio = audio.squeeze(0)
    window = torch.hann_window(n_fft, device=audio.device)
    spec = torch.stft(audio, n_fft=n_fft, hop_length=hop,
                      window=window, return_complex=True)   # (F, T)
    power = spec.abs().pow(2)                               # (F, T)
    # 只统计有声帧（能量 > 10% 峰值），避免静音帧主导频带分布
    frame_energy = power.sum(0)                             # (T,)
    voiced = frame_energy >= frame_energy.max() * 0.1
    pw = power[:, voiced]                                   # (F, Tv)
    if pw.shape[1] == 0:
        pw = power
    # log 频率分桶（低频桶细、高频桶粗，符合听觉）
    F = pw.shape[0]
    edges = torch.logspace(0, torch.log10(torch.tensor(F, dtype=torch.float32)),
                           n_bands + 1, device=audio.device).round().long().clamp(0, F - 1)
    bands = []
    for k in range(n_bands):
        if edges[k + 1] > edges[k]:
            bands.append(pw[edges[k]:edges[k + 1]].mean())
        else:
            bands.append(pw[edges[k]].mean())
    profile = torch.stack(bands)
    return profile / (profile.sum() + 1e-8)


print('Functions defined: differentiable_forward, token_level_mean, ecapa_embedding, spectral_band_profile')


## 5. 计算目标音色统计量

从数据集中随机选取 N 条音频，提取 WavLM 特征并取平均。

In [ ]:
N_REF = 20  # 参考音频数量（ECAPA 说话人嵌入用，10-30 足够）
N_REF_BAND = 100  # 频谱频带分布参考数量：更多样本→平均频谱质感更稳定
WAVLM_LAYER = 4
WAVLM_SR = 16000
KOKORO_SR = 24000
SEED = 42

# 随机选取参考音频
rng = np.random.RandomState(SEED)
n = min(N_REF, len(all_wavs))
idx = rng.choice(len(all_wavs), n, replace=False)
ref_files = [all_wavs[i] for i in idx]
print(f'Using {len(ref_files)} reference audio files')

# 提取目标说话人嵌入（ECAPA, 192 维）
embs = []
for p in ref_files:
    wav, _ = librosa.load(p, sr=WAVLM_SR)
    # 去静音段
    wav, _ = librosa.effects.trim(wav, top_db=30)
    wav_t = torch.from_numpy(wav).float().to(DEVICE)
    with torch.no_grad():
        e = ecapa_embedding(spk_enc, wav_t.unsqueeze(0))
    embs.append(e)
tgt_emb = F.normalize(torch.stack(embs).mean(dim=0), dim=0)
print(f'Target embedding: {tgt_emb.shape}, norm={tgt_emb.norm().item():.4f}')

# 目标频谱频带能量分布（尖细音色的高频占比）：用 100 条参考统计，
# 更多样本 → 平均频谱质感更稳定（20 条受单句内容影响大，与单句合成
# 有不可达 gap）
rng_band = np.random.RandomState(SEED + 1)
n_band = min(N_REF_BAND, len(all_wavs))
idx_band = rng_band.choice(len(all_wavs), n_band, replace=False)
band_profiles = []
for i_b in idx_band:
    wav_b, _ = librosa.load(all_wavs[i_b], sr=KOKORO_SR)
    wav_b, _ = librosa.effects.trim(wav_b, top_db=30)
    wav_b_t = torch.from_numpy(wav_b).float().to(DEVICE)
    with torch.no_grad():
        bp = spectral_band_profile(wav_b_t)
    band_profiles.append(bp)
tgt_band = torch.stack(band_profiles).mean(dim=0)
print(f'Target band profile ({len(band_profiles)} refs): '
      f'{[f"{v:.3f}" for v in tgt_band.tolist()]}')

## 6. 梯度反演训练

核心优化循环：style vector (256维) → 合成音频 → WavLM 特征 → 损失 → 反向传播。

In [ ]:
# 训练配置
INIT_VOICE = 'zf_xiaoyi'      # 初始 voicepack（acoustic 基准）
TARGET_F0_HZ = 349.8          # 数据集 F0 mean（Hz）
LR = 2e-4                # 学习率（论文最优）
MAX_STEPS = 2000         # 最大步数（通常 200-1000 步收敛）
THRESHOLD = 0.15         # 停止阈值 (cosine loss, 范围 [0,2], 完全匹配=0)
SPEED = 1.0

# 训练文本：优先使用数据集 manifest 中的真实文本（含标点）
# 关键：训练文本必须与数据集音频对应，模型才能学到标点静音规律。
# 若使用固定句子，合成音频的韵律/标点分布与参考音频不一致，
# 标点位置的静音特征无法通过 WavLM 全局统计量学到。
if MANIFEST_PATH:
    with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
        manifest_recs = [json.loads(l) for l in f if l.strip()]
    TEXTS = [r.get('text') or r.get('normalized_text', '') for r in manifest_recs]
    TEXTS = [t for t in TEXTS if t.strip()]
    print(f'[Dataset] Loaded {len(TEXTS)} texts from manifest (w/ punctuation)')
    for t in TEXTS[:3]:
        print(f'  {t}')
else:
    # fallback：无 manifest 时的固定文本
    TEXTS = [
        '主人，今天天气真好，我们去公园散步吧。',
        '知道啦知道啦，本小姐正在准备呢，不要催嘛。',
        '真正的强者，是不会被这种小事打倒的。',
        '嘿嘿，这个想法不错嘛，本小姐允许你继续说下去。',
        '有时候沉默比言语更有力量，你说是吧？',
        '夜晚的风带着一丝凉意，让人忍不住想要靠近温暖的东西。',
        '梦想这种东西，只要不放弃，总有一天会实现的。',
        '今天的咖啡有点苦，就像生活一样，但也别有一番风味。',
        '不要害怕失败，因为每一次跌倒都是成长的契机。',
        '本小姐的直觉一向很准，这次也不会例外的。',
    ]
    print(f'[Fallback] Using {len(TEXTS)} fixed texts')

# 预计算所有文本的音素（保留标点音素）
all_phonemes = []
for t in TEXTS:
    chunks = list(pipeline(t, voice='zf_xiaoyi'))
    if chunks:
        best = max(chunks, key=lambda c: len(c.phonemes) if c.phonemes else 0)
        if best.phonemes:
            all_phonemes.append(best.phonemes)
print(f'G2P: {len(all_phonemes)}/{len(TEXTS)} texts phonemized')
for i, ph in enumerate(all_phonemes[:3]):
    print(f'  [{i}] {repr(ph)}')


In [ ]:
# 初始化 style vector
torch.manual_seed(SEED)
np.random.seed(SEED)

# === 初始 style：官方音色（acoustic + prosody）===
# 数据集 F0=349.8 Hz 远超 Kokoro 中文预设范围（126-292 Hz），
# 用 8 个预设做 PCA 线性外推必然超出 prosody 训练分布（OOD），
# 导致 F0/N 预测异常、合成音频退化、WavLM loss 假低。
# 因此：
#   - 初始 prosody 直接用官方音色（分布内，稳定）
#   - 绝对音高由 F0 均值 loss 在训练中拉向目标（见下）
voices_f0 = {
    'zf_xiaoyi': 291.7, 'zm_yunxia': 291.3, 'zf_xiaoxiao': 236.9,
    'zf_xiaobei': 236.8, 'zf_xiaoni': 233.4, 'zm_yunxi': 171.1,
    'zm_yunyang': 129.6, 'zm_yunjian': 126.4,
}

init_voicepack = pipeline.load_voice(INIT_VOICE)
init_style_vec = init_voicepack[len(init_voicepack) // 2].flatten().numpy()
init_style = torch.from_numpy(init_style_vec).float().unsqueeze(0).to(DEVICE)
print(f'Init style: {INIT_VOICE} (acoustic + prosody), '
      f'norm={init_style.norm().item():.4f}')

# F0 均值目标比例：winefox F0 = 官方 F0 × ratio
F0_TARGET_RATIO = TARGET_F0_HZ / voices_f0[INIT_VOICE]
print(f'F0 target ratio: {F0_TARGET_RATIO:.3f} '
      f'({voices_f0[INIT_VOICE]} -> {TARGET_F0_HZ} Hz)')

# ============================================================
# 参考韵律预计算（官方音色）
# ============================================================
# 数据集音频本身就是 Kokoro 生成的（ref3_clean + speaker embedding），
# 因此官方音色的 prosody 输出（时长/语调/能量）就是"这段文本该怎么读"
# 的参考节奏。用它约束 winefox 音色的韵律，而非硬性规定"标点处静音"。
official_pack = pipeline.load_voice(INIT_VOICE)
official_style = torch.from_numpy(
    official_pack[len(official_pack) // 2].flatten().numpy()
).float().unsqueeze(0).to(DEVICE)
official_style.requires_grad_(False)

ref_prosody = {}   # phoneme -> dict(dur, f0, n, pd)
with torch.no_grad():
    for ph in all_phonemes:
        try:
            _, f0_r, n_r, dur_r, pd_r, ids = differentiable_forward(
                model, ph, official_style, SPEED
            )
            ref_prosody[ph] = {
                'dur': dur_r.detach(),   # 连续时长（round 前）
                'f0': f0_r.detach(),     # 帧级 F0
                'n': n_r.detach(),       # 帧级 N
                'pd': pd_r.detach(),     # 整数时长（用于 token 级化）
            }
        except Exception as e:
            print(f'  ref prosody failed for {ph[:20]!r}: {e}')
print(f'Ref prosody computed: {len(ref_prosody)}/{len(all_phonemes)}')

# ============================================================
# 联合优化：speaker loss → acoustic，韵律 loss → prosody
# ============================================================
# 两个 loss 在同一 step 内分别 backward，梯度隔离：
#   - speaker loss 是音色匹配（ECAPA embedding cosine），只作用于 acoustic（前128维）
#   - 韵律 loss 是时间维度匹配（时长/语调/能量 vs 官方参考），
#     依赖 s = ref_s[:, 128:]，天然只作用于 prosody（后128维）
W_DUR = 1.0     # 节奏（每个音素时长）权重
W_F0  = 0.03    # 语调（F0 轮廓）权重 —— 目标为缩放后的参考轮廓（含绝对音高）
W_N   = 1.0     # 能量（逐音素 N，标点处自然静音）权重
W_BAND = 5.0    # 频谱频带能量分布权重 —— 把音色质感（高频占比→尖细）拉向目标
W_NORM = 0.25   # acoustic norm 正则权重 —— 防止 acoustic 涨出分布（机械/大舌头）
MIN_STEPS = 200  # 最小步数：避免音色 loss 首轮达标时韵律未充分优化就停止

# 拆分为 acoustic / prosody 两个独立参数，可分别设学习率：
# spk 梯度路径长（decoder -> audio -> resample -> ECAPA-TDNN），
# 2e-4 下 1250 步几乎不动；prosody 路径短、收敛快。
LR_AC = 1e-3    # acoustic 学习率（spk loss 用，路径深需更大步长）
acoustic = init_style[:, :128].detach().clone().requires_grad_(True)
prosody = init_style[:, 128:].detach().clone().requires_grad_(True)
init_ac_norm = acoustic.norm().detach()   # acoustic norm 正则目标（分布内水平）
opt = torch.optim.Adam([
    {'params': [acoustic], 'lr': LR_AC},
    {'params': [prosody], 'lr': LR},
])

best_loss = float('inf')
best_style = None
no_improve = 0          # 连续无改善步数（early stopping）
EARLY_STOP_PATIENCE = 200  # 连续多少步（EMA 平滑后）总 loss 无改善则停止
EMA_ALPHA = 0.05        # total loss 的 EMA 平滑系数：spk 跨文本波动大，
                        # 用平滑值判定早停，避免韵律仍在改善时被 spk 波动误停
ema_total = None
t0 = time.time()

print(f'\nStarting joint optimization: lr={LR}, max_steps={MAX_STEPS}, '
      f'threshold={THRESHOLD}')
print(f'  speaker loss -> acoustic | prosody loss -> (dur={W_DUR}, f0={W_F0}, n={W_N})')
print('-' * 80)

for step in range(MAX_STEPS):
    # 旋转文本
    phonemes = all_phonemes[step % len(all_phonemes)]
    if phonemes not in ref_prosody:
        continue

    # Forward: style -> (audio, F0_pred, N_pred, duration, pred_dur, input_ids)
    ref_s = torch.cat([acoustic, prosody], dim=1)
    try:
        audio, f0_pred, n_pred, duration, pred_dur, input_ids = \
            differentiable_forward(model, phonemes, ref_s, SPEED)
    except Exception as e:
        print(f'  step {step}: forward failed: {e}')
        continue

    if audio.numel() == 0 or audio.abs().max() < 1e-6:
        print(f'  step {step}: audio empty/silent, skipping')
        continue

    # --- speaker loss (ECAPA embedding, acoustic 前128维) ---
    # 可微：syn_emb -> ECAPA -> audio -> decoder -> style[:128]
    audio_16k = torchaudio.functional.resample(
        audio.unsqueeze(0) if audio.dim() == 1 else audio,
        KOKORO_SR, WAVLM_SR
    )
    syn_emb = ecapa_embedding(spk_enc, audio_16k)   # (192,) 可微
    spk_loss = 1 - F.cosine_similarity(
        syn_emb.unsqueeze(0), tgt_emb.unsqueeze(0))

    # --- 频谱频带损失（acoustic）：高频占比拉向目标 → 尖细清脆 ---
    band_loss = F.l1_loss(spectral_band_profile(audio), tgt_band)
    # --- acoustic norm 正则：防止 acoustic 涨出 decoder 分布 ---
    norm_penalty = (acoustic.norm() - init_ac_norm) ** 2
    ac_loss = spk_loss + W_BAND * band_loss + W_NORM * norm_penalty

    # --- 韵律 loss (prosody 后128维, 时间维度) ---
    ref = ref_prosody[phonemes]

    # 1. 节奏：每个音素的连续时长匹配官方参考（round 前，可微）
    dur_loss = F.l1_loss(duration, ref['dur'])

    # 2. 语调：F0 轮廓匹配（帧级 → token 级）
    #    目标 = 官方参考轮廓 × F0_TARGET_RATIO：形状不变、绝对音高
    #    抬到 TARGET_F0_HZ。之前用"去均值轮廓 + 独立比例均值 loss"时
    #    二者互相拉扯（轮廓把均值拉回 291.7Hz，比例均值 loss 梯度又比
    #    轮廓 loss 小约 4 个数量级），实测 f0m 卡在 1.009 拉不动。
    #    单一目标同时约束形状与音高，方向一致。
    f0_pred_tok = token_level_mean(f0_pred, pred_dur)
    f0_ref_tok = token_level_mean(ref['f0'], ref['pd'])
    f0_target_tok = f0_ref_tok * F0_TARGET_RATIO
    f0_loss = F.l1_loss(f0_pred_tok, f0_target_tok)

    # 3. 能量：逐音素 N 匹配（标点 token 在参考中 N≈0，自然约束标点静音）
    n_pred_tok = token_level_mean(n_pred, pred_dur)
    n_ref_tok = token_level_mean(ref['n'], ref['pd'])
    # 纯 token 级：帧级对齐对时长误差敏感（prosody 移动→帧错位），
    # 且官方 N 帧级参考不是目标音色的，实测 n loss 不降反升、prosody 震荡
    n_loss = F.l1_loss(n_pred_tok, n_ref_tok)

    # 4. 诊断指标：当前 F0 均值 / 官方 F0 均值（目标为 F0_TARGET_RATIO）
    f0_mean_ratio = f0_pred_tok.mean() / (f0_ref_tok.mean() + 1e-6)

    prosody_loss = (W_DUR * dur_loss + W_F0 * f0_loss
                    + W_N * n_loss)

    # --- 分别 backward + 梯度隔离 ---
    opt.zero_grad()
    # 1. acoustic loss backward（spk + band + norm，保留计算图给 prosody loss）
    ac_loss.backward(retain_graph=True)
    # 清零 prosody 梯度：spk loss 经 audio 也依赖 F0/N（prosody），不应污染
    if prosody.grad is not None:
        prosody.grad.zero_()
    # 2. prosody loss backward：只依赖 s=ref_s[:,128:]（duration/F0/N），
    #    梯度天然只到 prosody，不会碰 acoustic.grad
    prosody_loss.backward()
    # 关键：此处绝不能清 acoustic 梯度——那会把 spk 给 acoustic 的梯度
    # 一起清掉，导致 acoustic 从不更新（此前 agrad 恒为 0、spk 永不下降
    # 的根因）。acoustic.grad 保持 spk_loss 的梯度，step 即完成双向隔离
    opt.step()

    total_loss = (spk_loss.item()
                  + W_DUR * dur_loss.item()
                  + W_F0 * f0_loss.item()
                  + W_N * n_loss.item())
    # EMA 平滑 total 再判定早停：spk loss 每步文本不同、波动约 ±0.15，
    # 直接比较会掩盖 f0/f0m 的持续改善（上一版 step 344 就误触发早停，
    # 当时 f0m 还在爬升）。
    if ema_total is None:
        ema_total = total_loss
    else:
        ema_total = EMA_ALPHA * total_loss + (1 - EMA_ALPHA) * ema_total
    if ema_total < best_loss:
        best_loss = ema_total
        best_style = torch.cat([acoustic, prosody], dim=1).detach().clone()
        no_improve = 0
    else:
        no_improve += 1

    if step % 50 == 0 or step == MAX_STEPS - 1:
        ac_grad = acoustic.grad.norm().item() if acoustic.grad is not None else 0
        pr_grad = prosody.grad.norm().item() if prosody.grad is not None else 0
        elapsed = time.time() - t0
        print(f'  step {step:5d} | spk={spk_loss.item():.4f} | band={band_loss.item():.4f} | '
              f'dur={dur_loss.item():.4f} | f0={f0_loss.item():.4f} | '
              f'f0m={f0_mean_ratio.item():.3f} | n={n_loss.item():.4f} | '
              f'agrad={ac_grad:.4f} | pgrad={pr_grad:.4f} | ({elapsed:.0f}s)')

    # Early stopping：连续 EARLY_STOP_PATIENCE 步总 loss 无改善则停止
    # 不用 WavLM 阈值：初始 acoustic 接近目标时 wl 首轮即达标，
    # 但韵律可能尚未收敛，按阈值停止会截断韵律优化。
    if step >= MIN_STEPS and no_improve >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stop: no improvement for {EARLY_STOP_PATIENCE} '
              f'steps at step {step}')
        break

if best_style is None:
    best_style = torch.cat([acoustic, prosody], dim=1).detach().clone()

elapsed = time.time() - t0
print(f'\nOptimization done in {elapsed:.0f}s')
print(f'Final loss: {best_loss:.4f}')
print(f'Style norm: {best_style.norm().item():.4f}')


## 7. 保存 voicepack

In [ ]:
# 保存为 (510, 1, 256) voicepack
final_voicepack = best_style.cpu().unsqueeze(0).expand(510, 1, 256).contiguous()
OUTPUT_PT = '/kaggle/working/winefox_voice.pt'
torch.save(final_voicepack, OUTPUT_PT)
print(f'Voicepack saved: {OUTPUT_PT}')
print(f'Shape: {final_voicepack.shape}')
print(f'Size: {os.path.getsize(OUTPUT_PT) / 1024:.1f} KB')

## 8. 验证

合成验证样本，并计算与参考音频的 WavLM 相似度。

In [ ]:
VERIFY_TEXTS = [
    '主人，今天天气真好，我们去公园散步吧。',
    '知道啦知道啦，本小姐正在准备呢。',
    '真正的强者，是不会被这种小事打倒的。',
    '嘿嘿，这个想法不错嘛，本小姐允许你继续说下去。',
    '夜晚的风带着一丝凉意，让人忍不住想要靠近温暖的东西。',
]

# 用 differentiable_forward 合成（与训练完全一致），绕开 KPipeline 的
# voice 参数：其 load_voice 只接受 str 或 CPU FloatTensor，传 CUDA
# tensor 会在 voice.split() 处报 TypeError。best_style 即 (1, 256)。
ref_s = best_style.to(DEVICE)

# 参考音频嵌入
rng2 = np.random.RandomState(123)
idx2 = rng2.choice(len(all_wavs), min(5, len(all_wavs)), replace=False)
ref_files_eval = [all_wavs[i] for i in idx2]

ref_embeddings = []
for rf in ref_files_eval:
    wav, _ = librosa.load(rf, sr=WAVLM_SR)
    wav, _ = librosa.effects.trim(wav, top_db=30)
    wav_t = torch.from_numpy(wav).float().to(DEVICE)
    with torch.no_grad():
        emb = ecapa_embedding(spk_enc, wav_t.unsqueeze(0))
    ref_embeddings.append(emb)

def cosine_sim(a, b):
    return torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# 合成验证样本
output_dir = '/kaggle/working/verify_output'
os.makedirs(output_dir, exist_ok=True)

print('=' * 60)
print('Verification samples')
print('=' * 60)

similarities = []
for i, text in enumerate(VERIFY_TEXTS):
    # G2P 取音素（pipeline 是 Cell 9 的 G2P pipeline，voice 用字符串即可）
    chunks = list(pipeline(text, voice='zf_xiaoyi'))
    if not chunks or not chunks[0].phonemes:
        print(f'[{i}] SKIP: G2P failed')
        continue
    phonemes = chunks[0].phonemes
    with torch.no_grad():
        audio, _, _, _, _, _ = differentiable_forward(
            model, phonemes, ref_s, speed=1.0)
    audio_np = audio.detach().cpu().numpy()
    if len(audio_np) == 0 or np.abs(audio_np).max() < 1e-6:
        print(f'[{i}] SKIP: silent audio')
        continue
    
    out_path = os.path.join(output_dir, f'verify_{i}.wav')
    sf.write(out_path, audio_np, KOKORO_SR)
    dur = len(audio_np) / KOKORO_SR
    
    # 相似度
    audio_16k = torchaudio.functional.resample(
        torch.from_numpy(audio_np).unsqueeze(0).float().to(DEVICE),
        KOKORO_SR, WAVLM_SR
    )
    with torch.no_grad():
        syn_emb = ecapa_embedding(spk_enc, audio_16k)
    
    sims = [cosine_sim(syn_emb, ref_emb) for ref_emb in ref_embeddings]
    avg_sim = np.mean(sims)
    similarities.append(avg_sim)
    
    print(f'[{i}] {out_path} ({dur:.1f}s) | sim={avg_sim:.4f} | {text}')

print('\n' + '=' * 60)
if similarities:
    avg = np.mean(similarities)
    print(f'Average ECAPA similarity: {avg:.4f}')
    print(f'  (>0.5 excellent, >0.3 usable, <0.2 needs work)')
print(f'Output dir: {output_dir}')

## 9. 导出 voices.bin

导出为 C++ 推理引擎使用的 voices.bin 格式。

In [ ]:
import struct

VOICE_NAME = 'winefox'
OUTPUT_BIN = '/kaggle/working/voices.bin'

# 读取 voicepack
voicepack = torch.load(OUTPUT_PT, map_location='cpu', weights_only=True)
style_arr = np.array(voicepack, dtype=np.float32).flatten()
print(f'Voicepack: shape={voicepack.shape}, dim={len(style_arr)}')

# 写入 voices.bin
with open(OUTPUT_BIN, 'wb') as f:
    # Header
    f.write(b'VOIC')
    f.write(struct.pack('<I', 1))          # version
    f.write(struct.pack('<I', 1))          # n_voices
    # Voice entry
    name_bytes = VOICE_NAME.encode('utf-8')
    f.write(struct.pack('<I', len(name_bytes)))
    f.write(name_bytes)
    f.write(struct.pack('<I', len(style_arr)))
    f.write(style_arr.tobytes())

size = os.path.getsize(OUTPUT_BIN)
print(f'voices.bin saved: {OUTPUT_BIN}')
print(f'Size: {size} bytes ({size/1024:.1f} KB)')
print(f'Voice name: {VOICE_NAME}')

## 10. 下载结果

从 `/kaggle/working/` 下载以下文件：
- `winefox_voice.pt` - PyTorch voicepack
- `voices.bin` - C++ 推理引擎格式
- `verify_output/verify_*.wav` - 验证音频

将 `voices.bin` 复制到本地 `models/tts/voices.bin` 即可在 WineFox 项目中使用。